<a href="https://colab.research.google.com/github/mariselvi242007/gen-ai-assignments/blob/main/05_RAG/RAG_Step3_Retrieving_Relevant_Context.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# RAG STEP 3: RETRIEVING RELEVANT CONTEXT
# ============================================================

!pip install -q chromadb sentence-transformers

import chromadb
from sentence_transformers import SentenceTransformer

print("=" * 60)
print("RAG STEP 3 - RETRIEVING RELEVANT CONTEXT")
print("=" * 60)


# ------------------------------------------------------------
# 1. Create document chunks
# ------------------------------------------------------------

chunks = [
    "Artificial Intelligence (AI) is a branch of computer science that focuses on creating systems that can perform tasks that normally require human intelligence.",

    "Machine Learning (ML) is a subset of Artificial Intelligence. Machine learning algorithms learn patterns from data and use those patterns to make predictions or decisions.",

    "Deep Learning is a subset of Machine Learning. It uses artificial neural networks with multiple layers to learn complex patterns from large amounts of data.",

    "Natural Language Processing (NLP) is a field of Artificial Intelligence that enables computers to understand, process, and generate human language.",

    "Generative AI is a type of artificial intelligence that can generate new content such as text, images, audio, video, and computer code.",

    "Retrieval-Augmented Generation (RAG) combines information retrieval with language generation. A RAG system first retrieves relevant information from a knowledge base and then uses that information to generate an answer."
]


# ------------------------------------------------------------
# 2. Load embedding model
# ------------------------------------------------------------

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("\nEmbedding model loaded!")


# ------------------------------------------------------------
# 3. Generate document embeddings
# ------------------------------------------------------------

document_embeddings = model.encode(chunks)


# ------------------------------------------------------------
# 4. Create ChromaDB
# ------------------------------------------------------------

client = chromadb.Client()

collection = client.get_or_create_collection(
    name="rag_retrieval"
)


# ------------------------------------------------------------
# 5. Store documents
# ------------------------------------------------------------

collection.add(
    ids=[
        f"chunk_{i}"
        for i in range(len(chunks))
    ],
    documents=chunks,
    embeddings=document_embeddings.tolist()
)

print("Documents stored in ChromaDB!")


# ------------------------------------------------------------
# 6. Create user query
# ------------------------------------------------------------

query = "What is machine learning?"

print("\n" + "-" * 60)
print("USER QUERY")
print("-" * 60)

print(query)


# ------------------------------------------------------------
# 7. Generate query embedding
# ------------------------------------------------------------

query_embedding = model.encode(
    [query]
)[0]


# ------------------------------------------------------------
# 8. Retrieve relevant documents
# ------------------------------------------------------------

results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=3
)


# ------------------------------------------------------------
# 9. Display retrieved context
# ------------------------------------------------------------

print("\n" + "-" * 60)
print("RETRIEVED RELEVANT CONTEXT")
print("-" * 60)

for i, (document, distance) in enumerate(
    zip(
        results["documents"][0],
        results["distances"][0]
    ),
    start=1
):

    print(f"\nResult {i}")
    print("Distance:", round(distance, 4))
    print("Context:")
    print(document)


# ------------------------------------------------------------
# 10. Combine retrieved context
# ------------------------------------------------------------

retrieved_context = "\n\n".join(
    results["documents"][0]
)

print("\n" + "-" * 60)
print("COMBINED CONTEXT")
print("-" * 60)

print(retrieved_context)


# ------------------------------------------------------------
# 11. Final result
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("RAG STEP 3 COMPLETED SUCCESSFULLY")
print("=" * 60)

print("Query:", query)
print("Number of retrieved documents:",
      len(results["documents"][0]))

print("\nThe relevant context has been successfully retrieved.")